# LLM-jp-4 33B ThinkingをGoogle Colabで4bit実行する

2026年8月18日に公開された **`llm-jp/llm-jp-4-33b-thinking`** を、Google Colab上でbitsandbytes NF4 4bit量子化して動かします。

このNotebookは **Colab Proの24GB級以上のGPUを第一ターゲット**にしています。33B denseモデルなので、無料版T4では通常のTransformers + 4bit実行はかなり厳しいと予想されます。

LLM-jp-4のThinkingモデルはOpenAI Harmony Response Formatを採用しており、`reasoning_effort`として`low` / `medium` / `high`を切り替えられる設計です。

```text
Google Colab
  ↓
GPU / VRAM確認
  ↓
Google Drive cache
  ↓
LLM-jp-4 33B ThinkingをNF4 4bitでロード
  ↓
Harmony chat template
  ↓
reasoning_effort切替
  ↓
日本語推論
  ↓
Gradio UI
  ↓
GPUメモリ確認
```


In [ ]:
# =========================================
# コード1 実行環境の確認
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U transformers accelerate bitsandbytes sentencepiece

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError('GPUランタイムを有効にしてください。')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('GPU:', GPU_NAME)
print('GPU memory: %.2f GB' % GPU_GB)

if GPU_GB < 20:
    print('WARNING: 33B denseの4bit実行にはVRAMがかなり厳しい可能性があります。')
    print('Colab Proの24GB級以上を推奨します。')


In [ ]:
# =========================================
# コード2 Google DriveとHugging Face cache
# =========================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LocalLLM')
CACHE_DIR = PROJECT_DIR / 'Program' / 'hf_cache_llmjp4_33b_thinking'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HUB_CACHE'] = str(CACHE_DIR)

print('CACHE_DIR:', CACHE_DIR)
print('Drive free: %.1f GB' % (shutil.disk_usage(CACHE_DIR).free/1024**3))


In [ ]:
# =========================================
# コード3 LLM-jp-4 33B ThinkingをNF4 4bitでロード
# =========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'llm-jp/llm-jp-4-33b-thinking'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()
INPUT_DEVICE = next(model.parameters()).device

print('model loaded:', MODEL_ID)
print('input device:', INPUT_DEVICE)
print('4bit:', getattr(model, 'is_loaded_in_4bit', False))
print('device map:', getattr(model, 'hf_device_map', 'N/A'))
print('GPU allocated: %.2f GB' % (torch.cuda.memory_allocated()/1024**3))
print('GPU reserved : %.2f GB' % (torch.cuda.memory_reserved()/1024**3))


## LLM-jp-4 ThinkingとHarmony

LLM-jp-4の`-thinking`モデルは、通常のLlama/Qwen形式のChatとは異なり **Harmony Response Format** を利用します。

モデルに同梱されたcustom tokenizer / chat templateを使うため、`trust_remote_code=True`が必要です。

このNotebookでは、まずモデル同梱の`apply_chat_template()`を利用し、生成されたHarmony形式のraw outputも確認できるようにしています。


In [ ]:
# =========================================
# コード4 Thinking生成関数
# =========================================
import gc
import torch

@torch.inference_mode()
def thinking_generate(
    prompt,
    reasoning_effort='medium',
    max_new_tokens=512,
    temperature=0.6,
):
    messages = [
        {'role': 'user', 'content': prompt}
    ]

    # LLM-jp-4のbundled chat templateを利用
    # reasoning_effortは low / medium / high
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        reasoning_effort=reasoning_effort,
    ).to(INPUT_DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=True,
        temperature=float(temperature),
        top_p=0.95,
        top_k=20,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_ids = outputs[0, inputs['input_ids'].shape[-1]:]

    # Harmony tokenを含む可能性があるため、まずraw decodeを返す
    raw_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False,
    ).strip()

    clean_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    del inputs, outputs, generated_ids
    gc.collect(); torch.cuda.empty_cache()

    return clean_text, raw_text


In [ ]:
# =========================================
# コード5 日本語Thinkingテスト
# =========================================
question = (
    '日本語で答えてください。'
    '9.11と9.8では、どちらの数が大きいですか？'
    '理由も説明してください。'
)

clean, raw = thinking_generate(
    question,
    reasoning_effort='medium',
    max_new_tokens=512,
)

print('=== CLEAN OUTPUT ===')
print(clean)
print('
=== RAW HARMONY OUTPUT ===')
print(raw)


In [ ]:
# =========================================
# コード6 reasoning_effortを比較
# =========================================
question = '日本語で、生成AIのハルシネーションが起きる理由を簡潔に説明してください。'

for effort in ['low', 'medium', 'high']:
    print('
' + '='*70)
    print('reasoning_effort =', effort)
    print('='*70)
    clean, _ = thinking_generate(
        question,
        reasoning_effort=effort,
        max_new_tokens=384,
        temperature=0.6,
    )
    print(clean)


## 生成token数について

Thinkingモデルでは推論部分が長くなることがあります。

最初は、

```text
low     : 256〜384
medium  : 384〜512
high    : 512〜1024
```

程度から試してください。33B denseモデルなので、Colabでは長いcontext・長い生成をいきなり使わない方が安全です。


In [ ]:
# =========================================
# コード7 Gradio Thinking UI
# =========================================
import gradio as gr

print('Gradio:', gr.__version__)

def gr_thinking(prompt, effort, max_tokens):
    if not (prompt or '').strip():
        return '質問を入力してください。', ''
    try:
        clean, raw = thinking_generate(
            prompt,
            reasoning_effort=effort,
            max_new_tokens=int(max_tokens),
            temperature=0.6,
        )
        return clean, raw
    except Exception as e:
        import traceback
        traceback.print_exc()
        msg = f'{type(e).__name__}: {e}'
        return msg, msg

with gr.Blocks(title='LLM-jp-4 33B Thinking') as demo:
    gr.Markdown('## LLM-jp-4 33B Thinking — 4bit / Harmony')
    prompt_box = gr.Textbox(
        label='Prompt',
        value='日本語で答えてください。AIのハルシネーションとは何ですか？',
        lines=5,
    )
    effort_box = gr.Radio(
        choices=['low', 'medium', 'high'],
        value='medium',
        label='reasoning_effort',
    )
    max_box = gr.Slider(128, 1024, value=512, step=64, label='max_new_tokens')
    run_btn = gr.Button('Generate', variant='primary')
    clean_box = gr.Textbox(label='Decoded output', lines=12)
    raw_box = gr.Textbox(label='Raw Harmony output', lines=12)
    run_btn.click(
        gr_thinking,
        inputs=[prompt_box, effort_box, max_box],
        outputs=[clean_box, raw_box],
        concurrency_limit=1,
    )

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=True, debug=False, show_error=True)


In [ ]:
# =========================================
# コード8 GPUメモリ確認
# =========================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()

print('GPU allocated: %.2f GB' % (torch.cuda.memory_allocated()/1024**3))
print('GPU reserved : %.2f GB' % (torch.cuda.memory_reserved()/1024**3))
free, total = torch.cuda.mem_get_info()
print('GPU free      : %.2f GB' % (free/1024**3))
print('GPU total     : %.2f GB' % (total/1024**3))


## 実験時に記録しておく項目

```text
GPU名 / VRAM
4bitロード成功・失敗
ロード直後のGPU allocated
reasoning_effort low / medium / highの違い
日本語の自然さ
Raw Harmony output
Gradio動作
最終GPU free memory
```

無料版T4でも試す場合は、ロードの成否をそのまま記録してください。成功しなくても、Proとの差を示す有用な結果になります。
